# ORI File Value Modifier

This notebook modifies the lower N% of values in a configurable ORI file by changing them to 9999.0.

**Features:**
- Load any ORI file from the data directories
- Configure what percentage (N%) of lowest values to modify
- Visualize the original and modified data
- Save the modified data to a new file
- Compatible with PEMTRON warpage analysis tool format

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import glob
from pathlib import Path
import ipywidgets as widgets
from IPython.display import display, clear_output
import pandas as pd

# Configure matplotlib for inline plotting
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 10

## Configuration

In [ ]:
# Configuration parameters
DATA_BASE_PATH = r"C:\Users\Lee\Desktop\Huni\PEMTRON_warpage\data"
ARTIFACT_VALUES = [9999.0, -9999.0, 99999.0, -99999.0, -4000.0]  # Values considered as artifacts
NEW_ARTIFACT_VALUE = 9999.0  # Value to replace lower N% with

# Default configuration
DEFAULT_PERCENTAGE = 10.0  # Default percentage of lowest values to modify
DEFAULT_OUTPUT_SUFFIX = "_modified"  # Suffix for output files

## File Discovery and Selection

In [ ]:
def find_ori_files(base_path):
    """
    Find all ORI files in the data directories.
    
    Args:
        base_path (str): Base path to search for ORI files
        
    Returns:
        list: List of ORI file paths
    """
    ori_patterns = [
        "**/*_ORI.txt",
        "**/*@_ORI.txt", 
        "**/*_ORI_A.txt"
    ]
    
    ori_files = []
    for pattern in ori_patterns:
        files = glob.glob(os.path.join(base_path, pattern), recursive=True)
        ori_files.extend(files)
    
    # Remove duplicates and sort
    ori_files = list(set(ori_files))
    ori_files.sort()
    
    return ori_files

def display_file_info(file_path):
    """
    Display information about the selected file.
    
    Args:
        file_path (str): Path to the file
    """
    if os.path.exists(file_path):
        file_size = os.path.getsize(file_path)
        file_size_mb = file_size / (1024 * 1024)
        
        print(f"File: {os.path.basename(file_path)}")
        print(f"Directory: {os.path.dirname(file_path)}")
        print(f"Size: {file_size_mb:.2f} MB ({file_size:,} bytes)")
        print(f"Full path: {file_path}")
    else:
        print(f"File not found: {file_path}")

# Find available ORI files
print("Searching for ORI files...")
ori_files = find_ori_files(DATA_BASE_PATH)
print(f"Found {len(ori_files)} ORI files:\n")

for i, file_path in enumerate(ori_files, 1):
    rel_path = os.path.relpath(file_path, DATA_BASE_PATH)
    print(f"{i:2d}. {rel_path}")

## Data Loading and Processing Functions

In [ ]:
def load_ori_data(file_path):
    """
    Load data from an ORI file.
    
    Args:
        file_path (str): Path to the ORI file
        
    Returns:
        numpy.ndarray: Loaded data array, or None if error
    """
    try:
        print(f"Loading data from: {os.path.basename(file_path)}")
        
        # Read the file
        with open(file_path, 'r', encoding='utf-8') as f:
            data = f.read()
        
        # Parse into numpy array
        data_lines = data.strip().split('\n')
        
        # Filter out empty lines and comments
        clean_lines = []
        for line in data_lines:
            line = line.strip()
            if line and not line.startswith('#') and not line.startswith('%'):
                try:
                    # Test if line can be converted to floats
                    float_values = [float(x) for x in line.split()]
                    if float_values:  # Non-empty list
                        clean_lines.append(line)
                except ValueError:
                    continue
        
        if not clean_lines:
            print("Error: No valid data lines found")
            return None
        
        # Convert to numpy array
        data_array = np.array([list(map(float, line.split())) for line in clean_lines])
        
        print(f"Loaded data shape: {data_array.shape}")
        print(f"Data type: {data_array.dtype}")
        
        return data_array
        
    except Exception as e:
        print(f"Error loading {file_path}: {e}")
        return None

def analyze_data(data_array, artifact_values=None):
    """
    Analyze the data array and provide statistics.
    
    Args:
        data_array (numpy.ndarray): Data to analyze
        artifact_values (list): List of values considered as artifacts
        
    Returns:
        dict: Analysis results
    """
    if artifact_values is None:
        artifact_values = ARTIFACT_VALUES
    
    # Create mask for non-artifact values
    non_artifact_mask = np.ones(data_array.shape, dtype=bool)
    for artifact_val in artifact_values:
        non_artifact_mask &= (data_array != artifact_val)
    
    valid_data = data_array[non_artifact_mask]
    
    total_points = data_array.size
    valid_points = valid_data.size
    artifact_points = total_points - valid_points
    
    analysis = {
        'total_points': total_points,
        'valid_points': valid_points,
        'artifact_points': artifact_points,
        'valid_percentage': (valid_points / total_points) * 100,
        'artifact_percentage': (artifact_points / total_points) * 100,
    }
    
    if valid_points > 0:
        analysis.update({
            'min_value': np.min(valid_data),
            'max_value': np.max(valid_data),
            'mean_value': np.mean(valid_data),
            'std_value': np.std(valid_data),
            'median_value': np.median(valid_data)
        })
    
    return analysis

def print_analysis(analysis):
    """
    Print analysis results in a formatted way.
    
    Args:
        analysis (dict): Analysis results from analyze_data
    """
    print("\n=== Data Analysis ===")
    print(f"Total data points: {analysis['total_points']:,}")
    print(f"Valid data points: {analysis['valid_points']:,} ({analysis['valid_percentage']:.1f}%)")
    print(f"Artifact points: {analysis['artifact_points']:,} ({analysis['artifact_percentage']:.1f}%)")
    
    if analysis['valid_points'] > 0:
        print(f"\nValid data statistics:")
        print(f"  Min value: {analysis['min_value']:.6f}")
        print(f"  Max value: {analysis['max_value']:.6f}")
        print(f"  Mean value: {analysis['mean_value']:.6f}")
        print(f"  Std deviation: {analysis['std_value']:.6f}")
        print(f"  Median value: {analysis['median_value']:.6f}")

## Value Modification Functions

In [ ]:
def modify_lower_values(data_array, percentage, artifact_values=None, new_value=9999.0):
    """
    Modify the lower N% of valid values in the data array.
    
    Args:
        data_array (numpy.ndarray): Original data array
        percentage (float): Percentage of lowest values to modify (0-100)
        artifact_values (list): List of values considered as artifacts (excluded from modification)
        new_value (float): Value to replace the lower N% with
        
    Returns:
        tuple: (modified_array, modification_info)
    """
    if artifact_values is None:
        artifact_values = ARTIFACT_VALUES
    
    # Create a copy of the original data
    modified_data = data_array.copy()
    
    # Create mask for non-artifact values
    non_artifact_mask = np.ones(data_array.shape, dtype=bool)
    for artifact_val in artifact_values:
        non_artifact_mask &= (data_array != artifact_val)
    
    valid_data = data_array[non_artifact_mask]
    
    if valid_data.size == 0:
        return modified_data, {
            'modified_count': 0,
            'threshold_value': None,
            'valid_points': 0,
            'error': 'No valid data points found'
        }
    
    # Calculate the threshold value
    threshold_percentile = percentage
    threshold_value = np.percentile(valid_data, threshold_percentile)
    
    # Create mask for values to modify (valid values below threshold)
    modification_mask = non_artifact_mask & (data_array <= threshold_value)
    
    # Count how many values will be modified
    modified_count = np.sum(modification_mask)
    
    # Apply the modification
    modified_data[modification_mask] = new_value
    
    modification_info = {
        'modified_count': modified_count,
        'threshold_value': threshold_value,
        'valid_points': valid_data.size,
        'percentage_requested': percentage,
        'percentage_actual': (modified_count / valid_data.size) * 100 if valid_data.size > 0 else 0,
        'new_value': new_value
    }
    
    return modified_data, modification_info

def print_modification_info(info):
    """
    Print modification information.
    
    Args:
        info (dict): Modification info from modify_lower_values
    """
    print("\n=== Modification Results ===")
    
    if 'error' in info:
        print(f"Error: {info['error']}")
        return
    
    print(f"Requested percentage: {info['percentage_requested']:.1f}%")
    print(f"Threshold value: {info['threshold_value']:.6f}")
    print(f"Values modified: {info['modified_count']:,}")
    print(f"Total valid points: {info['valid_points']:,}")
    print(f"Actual percentage modified: {info['percentage_actual']:.2f}%")
    print(f"New value assigned: {info['new_value']}")

## Visualization Functions

In [ ]:
def plot_data_comparison(original_data, modified_data, title_suffix=""):
    """
    Plot comparison between original and modified data.
    
    Args:
        original_data (numpy.ndarray): Original data array
        modified_data (numpy.ndarray): Modified data array
        title_suffix (str): Suffix for plot title
    """
    fig, axes = plt.subplots(2, 2, figsize=(15, 12))
    fig.suptitle(f'Data Comparison {title_suffix}', fontsize=16, fontweight='bold')
    
    # Original data heatmap
    im1 = axes[0,0].imshow(original_data, cmap='jet', aspect='auto')
    axes[0,0].set_title('Original Data')
    axes[0,0].set_xlabel('Column')
    axes[0,0].set_ylabel('Row')
    plt.colorbar(im1, ax=axes[0,0])
    
    # Modified data heatmap
    im2 = axes[0,1].imshow(modified_data, cmap='jet', aspect='auto')
    axes[0,1].set_title('Modified Data')
    axes[0,1].set_xlabel('Column')
    axes[0,1].set_ylabel('Row')
    plt.colorbar(im2, ax=axes[0,1])
    
    # Difference map
    diff_data = modified_data - original_data
    im3 = axes[1,0].imshow(diff_data, cmap='RdBu', aspect='auto')
    axes[1,0].set_title('Difference (Modified - Original)')
    axes[1,0].set_xlabel('Column')
    axes[1,0].set_ylabel('Row')
    plt.colorbar(im3, ax=axes[1,0])
    
    # Histogram comparison
    # Filter out artifact values for histogram
    orig_valid = original_data[(original_data != 9999.0) & (original_data != -9999.0) & 
                              (original_data != 99999.0) & (original_data != -99999.0) & 
                              (original_data != -4000.0)]
    mod_valid = modified_data[(modified_data != 9999.0) & (modified_data != -9999.0) & 
                             (modified_data != 99999.0) & (modified_data != -99999.0) & 
                             (modified_data != -4000.0)]
    
    if len(orig_valid) > 0 and len(mod_valid) > 0:
        axes[1,1].hist(orig_valid.flatten(), bins=50, alpha=0.7, label='Original', density=True)
        axes[1,1].hist(mod_valid.flatten(), bins=50, alpha=0.7, label='Modified', density=True)
        axes[1,1].set_title('Value Distribution (Valid Data Only)')
        axes[1,1].set_xlabel('Value')
        axes[1,1].set_ylabel('Density')
        axes[1,1].legend()
        axes[1,1].grid(True, alpha=0.3)
    else:
        axes[1,1].text(0.5, 0.5, 'No valid data for histogram', 
                      ha='center', va='center', transform=axes[1,1].transAxes)
        axes[1,1].set_title('Value Distribution')
    
    plt.tight_layout()
    plt.show()

def plot_single_data(data_array, title="Data Visualization"):
    """
    Plot a single data array.
    
    Args:
        data_array (numpy.ndarray): Data to plot
        title (str): Plot title
    """
    fig, axes = plt.subplots(1, 2, figsize=(15, 6))
    fig.suptitle(title, fontsize=16, fontweight='bold')
    
    # Heatmap
    im = axes[0].imshow(data_array, cmap='jet', aspect='auto')
    axes[0].set_title('2D Heatmap')
    axes[0].set_xlabel('Column')
    axes[0].set_ylabel('Row')
    plt.colorbar(im, ax=axes[0])
    
    # Histogram (excluding artifact values)
    valid_data = data_array[(data_array != 9999.0) & (data_array != -9999.0) & 
                           (data_array != 99999.0) & (data_array != -99999.0) & 
                           (data_array != -4000.0)]
    
    if len(valid_data) > 0:
        axes[1].hist(valid_data.flatten(), bins=50, alpha=0.7, color='blue', edgecolor='black')
        axes[1].set_title('Value Distribution (Valid Data Only)')
        axes[1].set_xlabel('Value')
        axes[1].set_ylabel('Frequency')
        axes[1].grid(True, alpha=0.3)
    else:
        axes[1].text(0.5, 0.5, 'No valid data for histogram', 
                    ha='center', va='center', transform=axes[1].transAxes)
        axes[1].set_title('Value Distribution')
    
    plt.tight_layout()
    plt.show()

## File Saving Functions

In [ ]:
def save_modified_data(data_array, original_file_path, suffix="_modified"):
    """
    Save modified data to a new file.
    
    Args:
        data_array (numpy.ndarray): Data to save
        original_file_path (str): Path of the original file
        suffix (str): Suffix to add to the filename
        
    Returns:
        str: Path to the saved file
    """
    # Generate output filename
    base_path, ext = os.path.splitext(original_file_path)
    output_path = f"{base_path}{suffix}{ext}"
    
    try:
        # Save data as text file with tab separation
        with open(output_path, 'w', encoding='utf-8') as f:
            for row in data_array:
                row_str = '\t'.join(f'{val:.1f}' for val in row)
                f.write(row_str + '\n')
        
        print(f"\n=== File Saved ===")
        print(f"Output file: {os.path.basename(output_path)}")
        print(f"Full path: {output_path}")
        print(f"File size: {os.path.getsize(output_path):,} bytes")
        
        return output_path
        
    except Exception as e:
        print(f"Error saving file: {e}")
        return None

## Interactive Interface

In [ ]:
# Create interactive widgets
if ori_files:
    # File selection dropdown
    file_options = [(os.path.relpath(f, DATA_BASE_PATH), f) for f in ori_files]
    file_selector = widgets.Dropdown(
        options=file_options,
        description='Select ORI File:',
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='500px')
    )
    
    # Percentage slider
    percentage_slider = widgets.FloatSlider(
        value=DEFAULT_PERCENTAGE,
        min=0.1,
        max=50.0,
        step=0.1,
        description='Percentage to modify:',
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='400px')
    )
    
    # Output suffix text
    suffix_text = widgets.Text(
        value=DEFAULT_OUTPUT_SUFFIX,
        description='Output suffix:',
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='300px')
    )
    
    # Buttons
    load_button = widgets.Button(
        description='Load & Analyze File',
        button_style='info',
        layout=widgets.Layout(width='150px')
    )
    
    modify_button = widgets.Button(
        description='Modify Data',
        button_style='warning',
        layout=widgets.Layout(width='150px'),
        disabled=True
    )
    
    save_button = widgets.Button(
        description='Save Modified Data',
        button_style='success',
        layout=widgets.Layout(width='150px'),
        disabled=True
    )
    
    # Output area
    output_area = widgets.Output()
    
    # Global variables to store data
    current_original_data = None
    current_modified_data = None
    current_file_path = None
    
else:
    print("No ORI files found in the data directory!")
    print(f"Please check that ORI files exist in: {DATA_BASE_PATH}")

In [ ]:
# Event handlers
def on_load_button_clicked(b):
    global current_original_data, current_file_path
    
    with output_area:
        clear_output()
        
        current_file_path = file_selector.value
        display_file_info(current_file_path)
        
        # Load data
        current_original_data = load_ori_data(current_file_path)
        
        if current_original_data is not None:
            # Analyze data
            analysis = analyze_data(current_original_data)
            print_analysis(analysis)
            
            # Plot data
            plot_single_data(current_original_data, 
                           f"Original Data: {os.path.basename(current_file_path)}")
            
            # Enable modify button
            modify_button.disabled = False
            save_button.disabled = True  # Disable save until modification
        else:
            modify_button.disabled = True
            save_button.disabled = True

def on_modify_button_clicked(b):
    global current_modified_data
    
    with output_area:
        clear_output(wait=True)
        
        if current_original_data is None:
            print("Please load data first!")
            return
        
        percentage = percentage_slider.value
        
        print(f"Modifying {percentage:.1f}% of lowest values...")
        
        # Perform modification
        current_modified_data, mod_info = modify_lower_values(
            current_original_data, percentage, ARTIFACT_VALUES, NEW_ARTIFACT_VALUE
        )
        
        # Print modification info
        print_modification_info(mod_info)
        
        # Show comparison
        plot_data_comparison(current_original_data, current_modified_data,
                           f"({percentage:.1f}% modified)")
        
        # Enable save button
        save_button.disabled = False

def on_save_button_clicked(b):
    with output_area:
        if current_modified_data is None:
            print("No modified data to save! Please modify data first.")
            return
        
        suffix = suffix_text.value
        output_path = save_modified_data(current_modified_data, current_file_path, suffix)
        
        if output_path:
            print("\nFile saved successfully!")
        else:
            print("\nError: Failed to save file.")

# Connect event handlers
if ori_files:
    load_button.on_click(on_load_button_clicked)
    modify_button.on_click(on_modify_button_clicked)
    save_button.on_click(on_save_button_clicked)

## Main Interface

Use the controls below to:
1. **Select** an ORI file from the dropdown
2. **Load & Analyze** the file to see its statistics and visualization
3. **Set percentage** of lowest values to modify using the slider
4. **Modify Data** to replace the lower N% of values with 9999.0
5. **Save** the modified data to a new file with your chosen suffix

In [ ]:
if ori_files:
    # Display the interface
    control_panel = widgets.VBox([
        widgets.HTML("<h3>File Selection</h3>"),
        file_selector,
        widgets.HTML("<h3>Modification Settings</h3>"),
        percentage_slider,
        suffix_text,
        widgets.HTML("<h3>Actions</h3>"),
        widgets.HBox([load_button, modify_button, save_button]),
        widgets.HTML("<h3>Results</h3>"),
        output_area
    ])
    
    display(control_panel)
else:
    print("\nNo ORI files available for processing.")
    print("Please ensure ORI files are present in the data directory.")

## Manual Processing (Alternative)

If you prefer to run the processing manually without the interactive interface, you can use the cells below:

In [ ]:
# Manual configuration - edit these values as needed
MANUAL_FILE_INDEX = 0  # Index of file to process (0-based)
MANUAL_PERCENTAGE = 15.0  # Percentage of lowest values to modify
MANUAL_OUTPUT_SUFFIX = "_modified_15pct"  # Output file suffix

print(f"Manual processing configuration:")
if ori_files and MANUAL_FILE_INDEX < len(ori_files):
    manual_file_path = ori_files[MANUAL_FILE_INDEX]
    print(f"File to process: {os.path.relpath(manual_file_path, DATA_BASE_PATH)}")
    print(f"Percentage to modify: {MANUAL_PERCENTAGE}%")
    print(f"Output suffix: {MANUAL_OUTPUT_SUFFIX}")
else:
    print("Invalid file index or no files available!")
    manual_file_path = None

In [ ]:
# Manual processing execution
if manual_file_path and os.path.exists(manual_file_path):
    print("Starting manual processing...\n")
    
    # Display file info
    display_file_info(manual_file_path)
    
    # Load data
    manual_original_data = load_ori_data(manual_file_path)
    
    if manual_original_data is not None:
        # Analyze original data
        manual_analysis = analyze_data(manual_original_data)
        print_analysis(manual_analysis)
        
        # Modify data
        print(f"\nModifying {MANUAL_PERCENTAGE}% of lowest values...")
        manual_modified_data, manual_mod_info = modify_lower_values(
            manual_original_data, MANUAL_PERCENTAGE, ARTIFACT_VALUES, NEW_ARTIFACT_VALUE
        )
        
        # Print modification results
        print_modification_info(manual_mod_info)
        
        # Visualize results
        plot_data_comparison(manual_original_data, manual_modified_data,
                           f"Manual Processing ({MANUAL_PERCENTAGE}% modified)")
        
        # Save results
        manual_output_path = save_modified_data(manual_modified_data, manual_file_path, MANUAL_OUTPUT_SUFFIX)
        
        if manual_output_path:
            print("\n✓ Manual processing completed successfully!")
        else:
            print("\n✗ Manual processing completed but failed to save file.")
    else:
        print("Failed to load data for manual processing.")
else:
    print("Cannot perform manual processing - invalid file path.")

## Usage Notes

**About the modification process:**
- Only **valid measurement values** are considered for modification (artifact values like 9999.0, -9999.0, etc. are excluded)
- The tool calculates the Nth percentile threshold from valid values and replaces all values at or below this threshold
- The modified values are replaced with 9999.0 (standard artifact value)
- Original artifact values remain unchanged

**File formats:**
- Input: ORI files with tab-separated numeric values
- Output: Same format as input, compatible with PEMTRON warpage analysis tool

**Visualization:**
- **Heatmaps** show the 2D data distribution
- **Histograms** show value distributions (excluding artifact values)
- **Difference maps** highlight modified regions

**Tips:**
- Start with small percentages (1-5%) to see the effect
- Use descriptive output suffixes to track different modification levels
- Review the statistics and visualizations before saving